# Problem Set 4 - Andrew Koren

## Problem 1. NSLS II Longitudinal Parameters

$$
\begin{gathered}
    {\ell \over \rho} = \theta = {2 \pi \over 60} = 0.104 \text{ rad}
\\  \rho = {p \over B q} \approx {E \over cBq}
\end{gathered}
$$

Each dipole is 2.62m

<!-- 1 T = 1 v s/m^2 -->

In [80]:
from scipy.constants import c, e, pi

E = 3 * 1e9 # eV
Bq = 0.4 # eV / (m^2/s)
rho = E/(c*Bq)

theta = 2*pi/60
dipole_len = theta*rho
dipole_len

2.619806277439602

b) just use $M_{0\rightarrow s}$ of a sector bend. We have $\rho \approx 25$ meters

$$
\begin{gathered}
\vec D_{0\rightarrow s} = \begin{bmatrix}
        \cos \theta & \rho \sin \theta & \rho (1 - \cos \theta)
    \\  -\sin \theta \over \rho & \cos \theta & \sin \theta
    \\  0 & 0 & 1
    \end{bmatrix} \begin{pmatrix} 0 \\ 0 \\ 1 \end{pmatrix}
\\  D(s) = 25 \left( 1- \cos \left(s \over 25\right)\right)
\\  D'(s) = \sin \left(s \over 25 \right)
\end{gathered}
$$
in meters / meters per second



c) use engineering formula

$$
U_L\left(\text{keV} \over \text{turn}\right) = {88.5 E^4[\text{GeV}] \over \rho[\text{m}]}
$$



In [ ]:
U_L = 88.5*(3**4)/(dipole_len)
print(U_L) # keV

2736.2710219192018


so 2.7 MeV per turn

d) 

$$
U_0 = eV_0 \sin \phi_s = 
$$

In [85]:
from numpy import sin
eV_0 = U_L/(sin(pi/6)) # keV
eV_0

np.float64(5472.542043838404)

yielding $5.4$ MV / $ 2.7$ MW

e)

$$
\Omega = \sqrt{ h \eta_c eV_0 \cos \phi_s \over 2\pi \beta c p_0}
$$

$$
\bar{D} = \alpha_c {R \over \rho}
\int s^2 / 50\rho
$$

## Problem 2. Dipole Edge Corrections - Horizontal

a)

Thin sector bend magnet:

$$
\begin{gathered}
    K = {1 \over \rho^2}
\\  M = \begin{bmatrix} 
        \cos \theta               & \rho \sin \theta 
    \\- {1 \over \rho}\sin \theta & \cos \theta 
    \end{bmatrix}

\end{gathered}
$$

Path through wedge:

$$
\tan(\alpha/2) = {z \over 2x}  
% = {1 - \cos \alpha \over \sin \alpha}
$$

$$
[B\rho] = {p \over q}
$$

b)

$$
\begin{gathered}
    \begin{bmatrix}
        1 & 0 \\ {\tan \alpha \over \rho} & 1
    \end{bmatrix}
    \begin{bmatrix} 
        \cos \theta               & \rho \sin \theta 
    \\- {1 \over \rho}\sin \theta & \cos \theta 
    \end{bmatrix}    \begin{bmatrix}
        1 & 0 \\ {\tan \alpha \over \rho} & 1
    \end{bmatrix}

\end{gathered}
$$

In [10]:
from sympy import *

theta, alpha, rho = symbols('theta alpha rho', real=True, positive=True)

Correction = Matrix([[1,0],[tan(alpha)/rho,1]])
Sector = Matrix([[cos(theta),rho*sin(theta)],[-1/rho*sin(theta),cos(theta)]])


def step_by_step_mul(operations: list):
    '''
    Matrix multiplication (xf) = [On,On-1,...,O2,O1](xi)
    '''
    O_r = operations[-1]
    for i in range(-1, -len(operations), -1):
            display(simplify(MatMul(*operations[0:i],O_r)))
            O_r = operations[i-1]*O_r
    result = expand(O_r)
    display(result)
    return result

M = step_by_step_mul([Correction,Sector,Correction])
M.subs((sin(theta)*tan(alpha)+cos(theta)),cos(theta-alpha)/cos(alpha))

Matrix([
[             1, 0],
[tan(alpha)/rho, 1]])*Matrix([
[     cos(theta), rho*sin(theta)],
[-sin(theta)/rho,     cos(theta)]])*Matrix([
[             1, 0],
[tan(alpha)/rho, 1]])

Matrix([
[             1, 0],
[tan(alpha)/rho, 1]])*Matrix([
[       sin(theta)*tan(alpha) + cos(theta), rho*sin(theta)],
[(-sin(theta) + cos(theta)*tan(alpha))/rho,     cos(theta)]])

Matrix([
[                                         sin(theta)*tan(alpha) + cos(theta),                     rho*sin(theta)],
[sin(theta)*tan(alpha)**2/rho - sin(theta)/rho + 2*cos(theta)*tan(alpha)/rho, sin(theta)*tan(alpha) + cos(theta)]])

Matrix([
[                                              cos(alpha - theta)/cos(alpha),                rho*sin(theta)],
[sin(theta)*tan(alpha)**2/rho - sin(theta)/rho + 2*cos(theta)*tan(alpha)/rho, cos(alpha - theta)/cos(alpha)]])

Let's do the bottom left corner by hand. We will show

$$
\begin{gathered}
        \frac{\sin{\left(\theta \right)} \tan^{2}{\left(\alpha \right)}}{\rho} 
    -   \frac{\sin{\left(\theta \right)}}{\rho} 
    +   \frac{2 \cos{\left(\theta \right)} \tan{\left(\alpha \right)}}{\rho}
    = - {\sin(\theta-2\alpha) \over \rho \cos^2(\alpha)}
% \\      \sin{\left(\theta \right)} \tan^{2}{\left(\alpha \right)}
%     -   \frac{\sin{\left(\theta \right)}}{\rho} 
%     +   \frac{2 \cos{\left(\theta \right)} \tan{\left(\alpha \right)}}{\rho}
%     = - {\sin(\theta-2\alpha) \over \rho \cos^2(\alpha)}
\end{gathered}
$$

<!-- $$
\sin(\theta - 2 \alpha) = 
$$ -->

$$
\begin{align*}
        -{\sin(\theta - 2 \alpha) \over \cos^2(\alpha)}
   &=   -{\sin(\theta)\cos(2\alpha) 
    -   \cos(\theta)\sin(2\alpha) \over \cos^2(\alpha)}
\\ &=   -{2\sin(\theta)\cos^2(\alpha)-\sin(\theta)
    -   2\cos(\theta)\sin(\alpha)\cos(\alpha) \over \cos^2  (\alpha)}
\\ &=   -2\sin(\theta)
    +   {\sin(\theta) \over \cos^2(\alpha)}
    +   2\cos(\theta)\tan(\alpha)
\\ &=   \sin(\theta)\left(-2
    +   {1\over \cos^2(\alpha)} \right)
    +   2\cos(\theta)\tan(\alpha)

\end{align*}
$$

All that is left now is to apply the identity
$$
        {\tan^{2}{\left(\alpha \right)}}
    -   1 
    = \sec^2(\alpha) 
    - 2
$$

----

<!-- $$
\begin{align*}
        \frac{\sin{\left(\theta \right)} \tan^{2}{\left(\alpha \right)}}{\rho} 
    -   \frac{\sin{\left(\theta \right)}}{\rho} 
    +   \frac{2 \cos{\left(\theta \right)} \tan{\left(\alpha \right)}}{\rho}
   &= {\sin(\theta) \over \rho} \left(\tan^2(\alpha)-1\right)
    + 
\end{align*}
$$ -->

Problem 3. Dispersion In Extraction Line

a) Since $D(s)\delta$ is an inhomogeneous solution, we know it's unique from difeq. I haven't taken difeq in a while though...

Let $D_1$ and $D_2$ be solutions to the differential equation 

$$
{d^2 D(s) \over d s^2} + K(s) D(s) = {1 \over \rho(s)}
$$

Then there exists a linear operator

$$
L(x) = {d^2 x\over ds^2} + K(s) x 
$$

such that $L(D_1) = L(D_2) = {1 \over \rho(s)}$.

Then for $D = D_1 - D_2$ we have

$$
L(D) = L(D_1 - D_2) = L(D_1) - L(D_2) = {1 \over \rho(s)} - {1 \over \rho(s)} = 0
$$

Since $L(D)=0$ we know that the sum of the two systems is not another solution to the differential equation and $D_2$ must be linearly dependent on $D_1$

b)

Let's find the 3x3 transfer matrix for the system. We'll use the matricies from the notes / Wiedemann textbook

In [88]:
d, f, R, l, d = symbols('d f R l d')
D, Dp = symbols('D_i D\'_i ')

drift = Matrix([
    [1, d, 0],
    [0, 1, 0],
    [0, 0, 1]
])

quad = Matrix([
    [1, 0, 0],
    [-1/f, 1, 0],
    [0, 0, 1]
])

theta = l/R
# sector = Matrix([
#     [cos(theta),R*sin(theta),R*(1-cos(theta))],
#     [-sin(theta)/R, cos(theta), sin(theta)],
#     [0, 0, 1]
# ])
sector = Matrix([
    [1,l,Rational(1,2)*l*theta],
    [0, 1, theta],
    [0, 0, 1]
])

M = step_by_step_mul([sector, quad, drift])
M = simplify(expand(M))
final = M * Matrix([D, Dp, 1])

Eq(Matrix([0,0,1]), final)

Matrix([
[1, l, l**2/(2*R)],
[0, 1,        l/R],
[0, 0,          1]])*Matrix([
[   1, 0, 0],
[-1/f, 1, 0],
[   0, 0, 1]])*Matrix([
[1, d, 0],
[0, 1, 0],
[0, 0, 1]])

Matrix([
[1, l, l**2/(2*R)],
[0, 1,        l/R],
[0, 0,          1]])*Matrix([
[   1,          d, 0],
[-1/f, (-d + f)/f, 0],
[   0,          0, 1]])

Matrix([
[1 - l/f, d - d*l/f + l, l**2/(2*R)],
[   -1/f,      -d/f + 1,        l/R],
[      0,             0,          1]])

Eq(Matrix([
[0],
[0],
[1]]), Matrix([
[D'_i*(d - d*l/f + l) + D_i*(f - l)/f + l**2/(2*R)],
[                    D'_i*(-d + f)/f - D_i/f + l/R],
[                                                1]]))

I sure am glad I chose to use the thin lens approximation. Let's let sympy do the hard work.

In [ ]:
Dsol1 = solve(final[0], D)[0]
Dpsol = solve(final[1].subs(D, Dsol1), Dp)[0]
display(simplify(Eq(Dpsol,Dp)))
# Dpsol

Dsol2 = solve(final[0].subs(Dp, Dpsol), D)[0]
display(simplify(Eq(Dsol2,D)))

Eq(D'_i, l*(-2*f + l)/(2*R*f))

Eq(D_i, l*(2*d*f - d*l + f*l)/(2*R*f))